# PaMCP E8 — Single-tool retrieval campaign

This notebook runs E8 on two separate tracks: ToolRet (heterogeneous retrieval with graded qrels) and the acquired MCPToolBench++ subset (single resolved primary target only).

**Protocol boundaries**
- No pooling of ToolRet and MCPToolBench++ metrics.
- Discovery sees queries and the full candidate corpus only. Evaluator qrels remain separate.
- MCPToolBench++ primary-target metrics are the main result; source-annotated similar_tools are evaluated only as a separate sensitivity analysis.
- MCP-Atlas trajectories are not exhaustive acceptable-tool-set gold. Its E8 diagnostic remains blocked until a validated identity/coverage mapping exists.
- B1/B1m are blocked because their source files are not in this repository snapshot.
- This experiment does not measure execution success, composition validity, observed effects or security.

The runner saves rankings per task, macro metrics, paired CIs/tests, Holm-adjusted p-values, candidate recall for cross-encoder reranking, plots, hashes, cache metadata and a ZIP. It is resume-safe when the same run ID, data, model revisions, device and parameters are reused.

In [ ]:
import os, sys, json, subprocess, pathlib, time, platform, shutil
REPO = "https://github.com/msellamiTN/taas-provenance-discovery.git"
BRANCH = "research/pamcp-multibenchmark-v1"
REMOTE_REF = f"refs/remotes/origin/{BRANCH}"
REPO_DIR = pathlib.Path("/content/taas-provenance-discovery")

def _git_run(args, *, check=False):
    result = subprocess.run(args, text=True, capture_output=True)
    if result.returncode != 0:
        print("GIT COMMAND FAILED:", " ".join(map(str, args)))
        print("exit code:", result.returncode)
        print("stderr:", (result.stderr or "<empty>")[-5000:])
        print("stdout:", (result.stdout or "<empty>")[-2000:])
    if check:
        result.check_returncode()
    return result

# First try to refresh an existing checkout. If its Git metadata/remote is stale,
# preserve it and clone a clean copy instead of hard-resetting or deleting user files.
checkout_ok = False
if REPO_DIR.exists() and (REPO_DIR / ".git").is_dir():
    remote = _git_run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO])
    fetch = _git_run([
        "git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin",
        f"refs/heads/{BRANCH}:{REMOTE_REF}"
    ]) if remote.returncode == 0 else remote
    if fetch.returncode == 0:
        checkout = _git_run([
            "git", "-C", str(REPO_DIR), "checkout", "-B", BRANCH, REMOTE_REF
        ])
        checkout_ok = checkout.returncode == 0
    else:
        print("Existing checkout refresh failed; attempting a clean clone.")
elif REPO_DIR.exists():
    print(f"{REPO_DIR} exists but is not a Git checkout; preserving it.")

if not checkout_ok:
    CLEAN_REPO_DIR = pathlib.Path("/content/taas-provenance-discovery-clean")
    if CLEAN_REPO_DIR.exists():
        shutil.rmtree(CLEAN_REPO_DIR)
    clone = _git_run([
        "git", "clone", "--depth", "1", "--single-branch",
        "--branch", BRANCH, REPO, str(CLEAN_REPO_DIR)
    ])
    if clone.returncode != 0:
        raise RuntimeError(
            "Could not refresh or clone the required PaMCP branch. "
            "Inspect the Git stderr printed above (network, DNS, GitHub access, or branch availability)."
        )
    REPO_DIR = CLEAN_REPO_DIR

os.chdir(REPO_DIR)
SRC_DIR = (REPO_DIR / "src").resolve()
if not (SRC_DIR / "taas" / "__init__.py").is_file():
    raise FileNotFoundError(f"TaaS src package not found: {SRC_DIR / 'taas'}")
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
REPO_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
EXPECTED_REMOTE_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
if REPO_COMMIT != EXPECTED_REMOTE_COMMIT:
    raise RuntimeError(f"Checkout mismatch: HEAD={REPO_COMMIT}, expected={EXPECTED_REMOTE_COMMIT}")
print("Repository:", REPO)
print("Branch:", BRANCH)
print("Commit:", REPO_COMMIT)

# Keep Colab's existing PyTorch installation; install the pinned project/model dependencies.
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-e",
    ".[test,experiment,benchmark-hf]", "sentence-transformers==5.1.2"
], check=True)

print("Python:", sys.version)
print("Platform:", platform.platform())

In [ ]:
subprocess.run([
    sys.executable, "-m", "pytest", "-q",
    "tests/test_e8_retrieval_campaign.py",
    "tests/test_toolret_adapter.py",
    "tests/test_mcp_atlas_adapter.py",
    "tests/test_mcpverse_adapter.py",
    "tests/test_evaluation_metrics.py",
    "tests/test_declared_effects_are_not_observations.py",
], check=True)

## 1. Acquire the pinned MCPToolBench++ subset

The source script fetches LiveMCPBench and MCPToolBench++ at the immutable commits in data/raw/SOURCES.txt. It reconstructs the local registry and the MCPToolBench++ task files. The E8 runner selects only single-primary-target examples for this track and records all exclusions.

In [ ]:
subprocess.run(["bash", "scripts/fetch_data.sh"], check=True)
sources = pathlib.Path("data/raw/SOURCES.txt").read_text(encoding="utf-8")
print(sources)
mcp_files = sorted(pathlib.Path("data/raw/mcptoolbenchpp").glob("*.json"))
assert mcp_files, "MCPToolBench++ raw JSON files were not acquired."
print("MCPToolBench++ category files:", [p.name for p in mcp_files])

## 2. Acquire ToolRet at its pinned revisions

ToolRet queries and tool corpus live in separate Hugging Face repositories. The candidate corpus is loaded independently from qrels; the adapter rejects qrels that refer to absent candidates. The revisions below are the previously recorded immutable snapshots.

In [ ]:
# Colab recovery: make the checkout's src-layout package importable even when
# this cell is run independently or the editable install did not expose `taas`.
from pathlib import Path
import os, sys
_candidate_roots = [Path.cwd(), Path('/content/taas-provenance-discovery')]
_repo_root = next((p.resolve() for p in _candidate_roots if (p / 'src' / 'taas' / 'experiments' / 'toolret_adapter.py').is_file()), None)
if _repo_root is None:
    raise RuntimeError('TaaS checkout not found. Run the Setup cell first and verify the repository clone.')
os.chdir(_repo_root)
_src_dir = str((_repo_root / 'src').resolve())
if _src_dir not in sys.path:
    sys.path.insert(0, _src_dir)

from huggingface_hub import HfApi
from taas.experiments.toolret_adapter import adapt_toolret_hf

TOOLRET_QUERY_REVISION = "b8c76ad3349ff17497b6bdb28bb5b8f61a0f6445"
TOOLRET_TOOL_REVISION = "e06c38c75612b6536bd959e08cdd345894aba6a7"

api = HfApi()
query_info = api.dataset_info("mangopy/ToolRet-Queries", revision=TOOLRET_QUERY_REVISION)
tool_info = api.dataset_info("mangopy/ToolRet-Tools", revision=TOOLRET_TOOL_REVISION)
assert query_info.sha == TOOLRET_QUERY_REVISION, query_info.sha
assert tool_info.sha == TOOLRET_TOOL_REVISION, tool_info.sha

toolret_manifest = adapt_toolret_hf(
    "data/processed/toolret",
    query_revision=TOOLRET_QUERY_REVISION,
    tool_revision=TOOLRET_TOOL_REVISION,
)
print(json.dumps(toolret_manifest["counts"], indent=2))
print("Data quality:", json.dumps(toolret_manifest.get("data_quality", {}), indent=2))
print("ToolRet query revision:", query_info.sha)
print("ToolRet tool revision:", tool_info.sha)

## 3. Check the two E8 tracks

The MCPToolBench++ corpus size should match the current repository subset (87 tools across 10 servers). The number of single-target tasks is computed by the runner after removing multiple-target, ambiguous and unresolved cases. ToolRet counts are read from its acquisition manifest; no historical aggregate is substituted.

In [ ]:
toolret_counts = toolret_manifest["counts"]
print("ToolRet:", json.dumps(toolret_counts, indent=2))

# Keep imports resilient if this cell is rerun after a kernel reset.
_SRC = str((pathlib.Path.cwd() / 'src').resolve())
if pathlib.Path(_SRC, 'taas').is_dir() and _SRC not in sys.path:
    sys.path.insert(0, _SRC)
from taas.data.registries import load_mcptoolbenchpp
mcp_tools, mcp_tasks = load_mcptoolbenchpp(pathlib.Path("data/raw"))
print("MCPToolBench++ tools:", len(mcp_tools))
print("MCPToolBench++ servers:", len({t.server for t in mcp_tools}))
print("MCPToolBench++ task records:", len(mcp_tasks))
assert len(mcp_tools) == 87, f"Expected pinned repository subset of 87 tools; found {len(mcp_tools)}"
assert len({t.server for t in mcp_tools}) == 10, "Unexpected MCPToolBench++ server count."

# Protocol-level preflight: compute exact E8 denominators and exclusions before ranking.
from experiments.e8_retrieval_campaign import (
    load_toolret_track, load_mcptoolbenchpp_track,
)
toolret_track = load_toolret_track("data/processed/toolret")
mcptoolbench_track = load_mcptoolbenchpp_track("data/raw")

print("ToolRet E8 audit:", json.dumps({
    "tasks": len(toolret_track.tasks),
    "candidate_tools": len(toolret_track.tools),
    "tasks_with_nonempty_qrels": sum(bool(toolret_track.qrels.get(t.task_id)) for t in toolret_track.tasks),
    "qrel_tasks": len(toolret_track.qrels),
    "unique_task_ids": len({t.task_id for t in toolret_track.tasks}),
    "unique_candidate_ids": len({t.tool_id for t in toolret_track.tools}),
}, indent=2))
print("MCPToolBench++ strict E8 audit:", json.dumps({
    "tasks_total": mcptoolbench_track.exclusions["tasks_total"],
    "tasks_evaluated_single_primary_target": mcptoolbench_track.exclusions["tasks_evaluated"],
    "tasks_excluded": mcptoolbench_track.exclusions["tasks_excluded"],
    "candidate_tools": len(mcptoolbench_track.tools),
    "servers": len({t.server_id for t in mcptoolbench_track.tools}),
    "exclusion_breakdown": mcptoolbench_track.exclusions,
}, indent=2))

assert len(toolret_track.tasks) == toolret_counts["tasks"] == 7960
assert len(toolret_track.tools) == toolret_counts["corpus_tools"] == 44453
assert all(toolret_track.qrels.get(t.task_id) for t in toolret_track.tasks), "ToolRet has tasks without non-empty qrels."
assert len(mcptoolbench_track.tasks) > 0, "No MCPToolBench++ single-target tasks survived strict filtering."
assert len(mcptoolbench_track.tools) == 87

## 4. Pin the retrieval model revisions

E5 uses query/passage prefixes. Cross-encoder reranking is limited to BM25 top-100 candidates and its candidate-recall ceiling is reported. This notebook runs on CPU by default; GPU access is optional and is not a precondition. The model revisions are resolved once and saved in the run manifest.


In [ ]:
import torch
import os
from huggingface_hub import HfApi

E5_MODEL = "intfloat/e5-small-v2"
CROSS_ENCODER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

e5_info = api.model_info(E5_MODEL)
cross_info = api.model_info(CROSS_ENCODER_MODEL)
E5_REVISION = e5_info.sha
CROSS_ENCODER_REVISION = cross_info.sha

# CPU is the fixed execution device for this campaign; GPU access is not required.
DEVICE = "cpu"
CPU_THREADS = max(1, min(4, os.cpu_count() or 1))
torch.set_num_threads(CPU_THREADS)

print("E5:", E5_MODEL, E5_REVISION)
print("Cross encoder:", CROSS_ENCODER_MODEL, CROSS_ENCODER_REVISION)
print("PyTorch:", torch.__version__)
print("Execution device:", DEVICE)
print("CPU logical cores:", os.cpu_count())
print("PyTorch CPU threads:", torch.get_num_threads())
assert DEVICE == "cpu"


## 5. Execute E8

The campaign runs on CPU and stores its run directory on persistent Google Drive. GPU access is not required. The run manifest records `device=cpu` and the CPU thread configuration.

The Cross-Encoder saves one atomic checkpoint per 16 query tasks (up to 1,600 query-tool pairs). CPU execution may take substantially longer than GPU execution. After a Colab interruption, rerun this cell with the same active run ID and matching data/model/device configuration. Completed batches are validated and reused; only the interrupted batch and subsequent batches are recomputed. A change in the experiment runner code, source snapshots, model revisions, device, top-k or seed requires a new run.


In [ ]:
# Durable E8 run state on Google Drive.
# CPU is the required execution device; no CUDA/GPU check is performed.
from google.colab import drive
import hashlib
import torch
import os

DEVICE = "cpu"
CPU_THREADS = max(1, min(4, os.cpu_count() or 1))
torch.set_num_threads(CPU_THREADS)
print("Execution device:", DEVICE)
print("PyTorch:", torch.__version__)
print("CPU logical cores:", os.cpu_count())
print("PyTorch CPU threads:", torch.get_num_threads())

drive.mount("/content/drive", force_remount=False)

OUTPUT_ROOT = pathlib.Path("/content/drive/MyDrive/PaMCP/E8/runs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ID_STATE = OUTPUT_ROOT / "active_run.json"

run_config = {
    "device": DEVICE,
    "cpu_threads": torch.get_num_threads(),
    "e5_model": E5_MODEL,
    "e5_revision": E5_REVISION,
    "cross_encoder_model": CROSS_ENCODER_MODEL,
    "cross_encoder_revision": CROSS_ENCODER_REVISION,
    "top_k": 10,
    "seed": 20261105,
}
run_config_key = hashlib.sha256(
    json.dumps(run_config, sort_keys=True).encode("utf-8")
).hexdigest()

RUN_ID = os.environ.get("E8_RUN_ID")
if not RUN_ID and RUN_ID_STATE.is_file():
    try:
        saved_state = json.loads(RUN_ID_STATE.read_text(encoding="utf-8"))
        saved_id = saved_state.get("run_id")
        saved_dir = OUTPUT_ROOT / str(saved_id)
        if (
            saved_state.get("config_key") == run_config_key
            and saved_id
            and not (saved_dir / "MANIFEST.json").is_file()
        ):
            RUN_ID = str(saved_id)
            print("Resuming compatible run:", RUN_ID)
    except (json.JSONDecodeError, OSError, AttributeError):
        print("Ignoring invalid prior active-run state.")

if not RUN_ID:
    RUN_ID = "E8-colab-cpu-" + time.strftime("%Y%m%dT%H%M%S", time.gmtime())
    print("Starting new CPU run:", RUN_ID)

os.environ["E8_RUN_ID"] = RUN_ID
RUN_ID_STATE.write_text(
    json.dumps({"run_id": RUN_ID, "config_key": run_config_key}, indent=2) + "\n",
    encoding="utf-8",
)

RUN_DIR = OUTPUT_ROOT / RUN_ID
if (RUN_DIR / "MANIFEST.json").is_file():
    print("This run is already finalized and immutable; loading its manifest:", RUN_DIR)
else:
    command = [
        sys.executable, "experiments/e8_retrieval_campaign.py",
        "--root", str(REPO_DIR),
        "--output-root", str(OUTPUT_ROOT),
        "--toolret-dir", "data/processed/toolret",
        "--raw-root", "data/raw",
        "--e5-model", E5_MODEL,
        "--e5-revision", E5_REVISION,
        "--cross-encoder-model", CROSS_ENCODER_MODEL,
        "--cross-encoder-revision", CROSS_ENCODER_REVISION,
        "--device", DEVICE,
        "--top-k", "10",
        "--seed", "20261105",
        "--run-id", RUN_ID,
    ]
    print("Run ID:", RUN_ID)
    print("Persistent output root:", OUTPUT_ROOT)
    print("Command:", " ".join(command))
    subprocess.run(command, check=True)

manifest_path = RUN_DIR / "MANIFEST.json"
if not manifest_path.is_file():
    raise RuntimeError(
        f"E8 did not produce a final manifest. The campaign was interrupted or failed: {RUN_DIR}"
    )

e8_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
print(json.dumps({
    "run_id": e8_manifest["run_id"],
    "status": e8_manifest["status"],
    "execution_scope_status": e8_manifest["execution_scope_status"],
    "device": e8_manifest.get("device"),
    "tracks": {
        name: {
            "tasks": item["task_count_discovery"],
            "tools": item["tool_count_candidates"],
            "baseline_ids": item["baseline_ids_executed"],
            "cache_hits": item["cache_hits"],
            "cross_encoder_checkpoint": item.get("cross_encoder_checkpoint"),
        }
        for name, item in e8_manifest["tracks"].items()
    },
    "blocked_or_not_comparable": e8_manifest["blocked_or_not_comparable"],
}, indent=2))


## 6. Inspect metrics and plots

The ranking files preserve each task's top-10 list. The per-task CSVs and paired-test JSON files retain denominators, effect estimates, bootstrap intervals, raw p-values and Holm-adjusted p-values. MCPToolBench++ also has a separate sensitivity report that includes annotated similar_tools.

In [ ]:
# Metrics are available only after the execution cell completed successfully.
if "e8_manifest" not in globals() or "RUN_DIR" not in globals():
    raise RuntimeError(
        "E8 metrics are unavailable: the execution cell did not complete. "
        "Return to section 5, verify Drive, and complete the CPU campaign first."
    )

for track_id, info in e8_manifest["tracks"].items():
    print("\\nTRACK:", track_id)
    metrics_path = RUN_DIR / info["metrics_file"]
    if not metrics_path.is_file():
        raise FileNotFoundError(f"Expected metrics file is missing: {metrics_path}")
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    for baseline, entry in metrics.items():
        overall = entry["metrics"]["overall"]["metrics"]
        print(
            baseline,
            "Hit@1=", overall["1"]["hit@1"]["value"],
            "Hit@5=", overall["5"]["hit@5"]["value"],
            "MRR@10=", overall["10"]["mrr@10"]["value"],
            "macro_Hit@1=", entry["macro_hit@1"]["value"],
        )
    if info.get("similar_tools_sensitivity_metrics"):
        print("Similar-tools sensitivity:", info["similar_tools_sensitivity_metrics"])
    print("Plots:", info["plots"])


## 7. Export the immutable run bundle

The ZIP contains the manifest, input fingerprints, cached rankings, per-task metric files, paired tests, plots and hash inventory. Keep the ZIP with the exact run ID and model revisions when reporting results.

In [ ]:
# Export only a completed immutable E8 run.
if "RUN_ID" not in globals() or "RUN_DIR" not in globals() or "e8_manifest" not in globals():
    raise RuntimeError(
        "No completed E8 campaign is available to export. The execution cell must finish first."
    )

archive_path = OUTPUT_ROOT / f"{RUN_ID}.zip"
if not archive_path.is_file():
    raise FileNotFoundError(
        f"Final E8 archive is missing although the manifest exists: {archive_path}"
    )
print("Run ID:", RUN_ID)
print("Manifest:", manifest_path)
print("Archive:", archive_path)
try:
    from google.colab import files
    files.download(str(archive_path))
except ImportError:
    print("Not running in Google Colab. Retrieve the archive from:", archive_path)
